In [1]:
import numpy as np
import awkward as ak
import matplotlib.pyplot as plt
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema

In [2]:
file = "root://cms-xrd-global.cern.ch//store/mc/RunIII2024Summer24NanoAODv15/WHToAA-AToBB-AToGG_Par-M-35_TuneCP5_13p6TeV_madgraph-pythia8/NANOAODSIM/150X_mcRun3_2024_realistic_v2-v2/120000/07822304-a19c-4b7d-ae7b-0f8b068bbe0e.root"
factory = NanoEventsFactory.from_root(
    f"{file}:Events",
    schemaclass=NanoAODSchema,
)
events = factory.events()

In [3]:
events.fields

['run',
 'GenVtx',
 'LHEReweightingWeight',
 'GenProton',
 'LHEPdfWeight',
 'GenDressedLepton',
 'BeamSpot',
 'LHE',
 'HLTriggerFinalPath',
 'SV',
 'L1Reco',
 'event',
 'SoftActivityJetNjets5',
 'boostedTau',
 'SoftActivityJetHT10',
 'Rho',
 'FatJetPFCand',
 'PFMET',
 'RawPuppiMET',
 'OtherPV',
 'MC',
 'genWeight',
 'genTtbarId',
 'FatJet',
 'GenPart',
 'SoftActivityJetNjets10',
 'Pileup',
 'PFCand',
 'TrackGenJetAK4',
 'GenMET',
 'CorrT1METJet',
 'GenJet',
 'TrkMET',
 'Electron',
 'TauProd',
 'IsoTrack',
 'LHEPart',
 'PV',
 'CaloMET',
 'bunchCrossing',
 'luminosityBlock',
 'SubGenJetAK8',
 'PVBS',
 'GenVisTau',
 'SoftActivityJetNjets2',
 'FiducialMET',
 'orbitNumber',
 'Jet',
 'Tau',
 'SoftActivityJetHT',
 'FsrPhoton',
 'LHEWeight',
 'LowPtElectron',
 'HLTriggerFirstPath',
 'TrigObj',
 'LHEScaleWeight',
 'HLT',
 'GenIsolatedPhoton',
 'DeepMETResolutionTune',
 'DeepMETResponseTune',
 'GenJetAK8',
 'SoftActivityJetHT5',
 'Photon',
 'PSWeight',
 'L1simulation',
 'HTXS',
 'PuppiMET',
 'So

In [19]:
def has_tau_ancestor(genparts, idx):
    """
    Returns True if GenPart[idx] has a tau ancestor.
    """

    while idx >= 0:

        pdgid = abs(int(genparts.pdgId[idx]))

        if pdgid == 15:
            return True

        idx = int(genparts.genPartIdxMother[idx])

    return False

def classify_reco_leptons(leptons, genparts):
    """
    Returns a jagged boolean array with the same structure
    as the reco leptons.
    """

    out = []

    for iev in range(len(leptons)):

        gp = genparts[iev]

        event_flags = []

        for lep in leptons[iev]:

            idx = int(lep.genPartIdx)

            if idx < 0:
                event_flags.append(False)
            else:
                event_flags.append(
                    has_tau_ancestor(gp, idx)
                )

        out.append(event_flags)

    return ak.Array(out)

In [17]:
electrons = events.Electron
muons = events.Muon

In [11]:
electrons.pt*1

<Array [[], [62.7], ..., [35.3, 19.3, 5.96]] type='30000 * var * float32[pa...'>

In [20]:
ele_from_tau_stage0 = classify_reco_leptons(electrons, events.GenPart)
mu_from_tau_stage0 = classify_reco_leptons(muons, events.GenPart)

In [60]:
mu_from_tau_stage0

<Array [[True], [False], [], ..., [], [], [False]] type='30000 * var * bool'>

In [61]:
ele_idx = ak.local_index(ele_from_tau_stage0)[ele_from_tau_stage0]

In [65]:
ele_idx[40:50]

<Array [[], [], [], [0], [], [], [], [], [], []] type='10 * var * int64'>

In [68]:
electrons[43][0].genPartIdx

36

In [24]:
muons.genPartIdx

<Array [[25], [41], [], ..., [], [], [-1]] type='30000 * var * int16[parame...'>

In [69]:
temp = events.GenPart[43]

In [50]:
temp

<GenParticleArray [GenParticle, ...] type='33 * GenParticle[genPartIdxMothe...'>

In [71]:
temp[events.GenPart[43][36].genPartIdxMother].fields

['genPartIdxMother',
 'statusFlags',
 'pdgId',
 'status',
 'eta',
 'mass',
 'phi',
 'pt',
 'iso',
 'genPartIdxMotherG',
 'distinctParentIdxG',
 'childrenIdxG',
 'distinctChildrenIdxG',
 'distinctChildrenDeepIdxG']

In [22]:
ele_from_tau_stage0

<Array [[], [False], ..., [False, False, False]] type='30000 * var * bool'>

In [78]:
n_true = ak.sum(ele_from_tau_stage0, axis=1)

In [80]:
n_true

<Array [0, 0, 0, 0, 0, 0, 0, 0, ..., 0, 0, 0, 0, 0, 0, 0] type='30000 * int64'>

In [81]:
mask = n_true > 1

print(ak.sum(mask))          # Number of such events
print(ak.where(mask)[0])     # Event indices

12
[517, 969, 2026, 7674, 9958, 17478, ..., 19535, 21170, 21914, 26480, 28968]


In [82]:
n_events = ak.sum(ak.any(ele_from_tau_stage0, axis=1))

In [83]:
n_events

1504

In [77]:
ak.sum(ele_from_tau_stage0)

1516